Heritage Skills SA, an NPO that conducts field-training programs across South Africa (e.g., Gauteng, Western Cape, KwaZulu-Natal) to train community members, research assistants, and municipal workers in archeological site documentation, spatial mapping (GIS), and heritage database cataloguing.

The Problem: The training coordinators record attendance, assessment scores, equipment allocations, and participant feedback using manual spreadsheets. The raw dataset contains severe human errors: inconsistent text casing, mixed date formats, missing IDs, duplicate rows, and bad numerical ranges.

In [6]:
# load and view data
import pandas as pd
heritage = pd.read_csv('dirty_npo_training_data.csv - raw_data.csv', index_col=0)
heritage.head()

,Full_Name,Province_Code,Training_Module,Start_Date,End_Date,Attendance_Pct,Pre_Test_Score,Post_Test_Score,Equipment_Assigned,Equipment_Returned,Funding_Sponsor
Participant_ID,,,,,,,,,,,
EMP-001,Sibusiso Dlamini,GP,GPR_Spatial_Mapping,2026/01/10,2026-01-24,95%,45.0,88,GPS_Unit_A1,Yes,SAHRA_Grant
EMP-002,SARAH VAN DER MERWE,WC,Arch_Site_Cataloguing,12/01/2026,2026-01-26,80%,52.0,79,Tablet_T04,YES,SAHRA_Grant
EMP-003,Thabo Mokoena,GP,GPR_Spatial_Mapping,2026-02-01,2026-02-15,100%,30.0,92,GPS_Unit_A2,Yes,DSAC_Fund
EMP-004,Lindiwe Khumalo,KZN,Community_Arch_Survey,2026-02-05,2026-02-19,40%,60.0,55,Dumpy_Level_01,No,DSAC_Fund
EMP-005,sibusiso dlamini,GP,GPR_Spatial_Mapping,2026/01/10,2026-01-24,95%,45.0,88,GPS_Unit_A1,Yes,SAHRA_Grant


In [7]:
import numpy as np
import pandas as pd

# 1. Standardize Text Casing
heritage['Full_Name'] = heritage['Full_Name'].astype(str).str.title().str.strip()

# 2. Standardize Province Codes
province_map = {'Gauteng': 'GP',
                'Western Cape': 'WC', 
                'GP': 'GP', 
                'WC': 'WC', 
                'KZN': 'KZN'}
heritage['Province_Code'] = heritage['Province_Code'].map(province_map)

# 3. Clean Equipment Returned Status
heritage['Equipment_Returned'] = heritage['Equipment_Returned'].astype(str).str.capitalize().replace(
    {'Yes': 'Returned',
     'No': 'Unreturned',
     'Missing': 'Unreturned'})

# 4. Clean Numeric Percentage Scores
heritage['Attendance_Pct'] = heritage['Attendance_Pct'].astype(str).str.rstrip('%')
heritage['Attendance_Pct'] = pd.to_numeric(heritage['Attendance_Pct'], errors='coerce')
heritage['Attendance_Pct'] = heritage['Attendance_Pct'].apply(lambda x: np.nan if (x > 100 or x < 0) else x)

# 5. Clean Test Scores
heritage['Pre_Test_Score'] = pd.to_numeric(heritage['Pre_Test_Score'], errors='coerce')
heritage['Post_Test_Score'] = pd.to_numeric(heritage['Post_Test_Score'], errors='coerce')

# Cap scores between 0 and 100
heritage['Post_Test_Score'] = heritage['Post_Test_Score'].clip(lower=0, upper=100)

# 6. Remove Duplicates
heritage = heritage.drop_duplicates(subset=['Full_Name', 'Start_Date'], keep='first')

# Output cleaned dataset
heritage.to_csv('cleaned_npo_training_data.csv', index=False)
heritage.head(10)

,Full_Name,Province_Code,Training_Module,Start_Date,End_Date,Attendance_Pct,Pre_Test_Score,Post_Test_Score,Equipment_Assigned,Equipment_Returned,Funding_Sponsor
Participant_ID,,,,,,,,,,,
EMP-001,Sibusiso Dlamini,GP,GPR_Spatial_Mapping,2026/01/10,2026-01-24,95.0,45.0,88,GPS_Unit_A1,Returned,SAHRA_Grant
EMP-002,Sarah Van Der Merwe,WC,Arch_Site_Cataloguing,12/01/2026,2026-01-26,80.0,52.0,79,Tablet_T04,Returned,SAHRA_Grant
EMP-003,Thabo Mokoena,GP,GPR_Spatial_Mapping,2026-02-01,2026-02-15,100.0,30.0,92,GPS_Unit_A2,Returned,DSAC_Fund
EMP-004,Lindiwe Khumalo,KZN,Community_Arch_Survey,2026-02-05,2026-02-19,40.0,60.0,55,Dumpy_Level_01,Unreturned,DSAC_Fund
EMP-006,Johan Botha,WC,Arch_Site_Cataloguing,2026-02-10,2026-02-24,NaN,55.0,0,Tablet_T02,Returned,Private_Donor
EMP-007,Nokuthula Zungu,KZN,Community_Arch_Survey,2026-03-01,2026-03-15,88.0,NaN,82,GPS_Unit_B1,Returned,DSAC_Fund
EMP-008,Kagiso Lesedi,GP,GPR_Spatial_Mapping,2026/03/05,2026/03/19,NaN,40.0,95,Tablet_T08,Returned,SAHRA_Grant
EMP-009,Pieter Naude,WC,Arch_Site_Cataloguing,01-03-2026,2026-03-15,75.0,48.0,70,Tablet_T09,Unreturned,Private_Donor
EMP-010,Nomvula Ndlovu,KZN,Community_Arch_Survey,2026-03-10,2026-03-24,90.0,50.0,85,GPS_Unit_B2,Returned,DSAC_Fund
